# Set up


In [ ]:
# Mount Google Drive so datasets, checkpoints, and outputs can be accessed from Colab.
# Paths containing Ablation_test below are legacy Google Drive folder names, not Git branches.
# Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone the unified public thesis repository into the local path expected by later cells.
!rm -rf /content/NeuS_thesis
!git clone https://github.com/CowboyScarlatto42/Thesis.git /content/NeuS_thesis
!cd /content/NeuS_thesis && git branch --show-current


In [ ]:
# Install the Python dependencies required by preprocessing, training, and postprocessing.
# Dipendenze principali
!pip install -q trimesh
!pip install -q opencv-python
!pip install -q plyfile
!pip install -q scipy
!pip install -q numpy

# Per visualizzazione
!pip install -q open3d
!pip install -q plotly
!pip install -q matplotlib


In [ ]:
# Install the tree utility used to inspect folder structures in Colab.
!apt-get -qq install tree

In [ ]:
# Print the legacy Ablation_test Google Drive folder tree for a quick dataset inspection.
!tree /content/drive/MyDrive/Tesi/neus/Ablation_test/

# Dataset

Dataset preparation.


In [ ]:
# Convert CORTO source data into the SPE3R-like intermediate format.
## CORTO to SPE3R
!python /content/NeuS_thesis/custom_codes/preprocessing/corto_to_spe3r_data.py \
  --images_in /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/img \
  --masks_in /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/masks \
  --geometry /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/geometry.json \
  --output_dir /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_spe3r


### Ablation without COLMAP

Ablation path using GT poses instead of COLMAP poses.


In [ ]:
# Build the NeuS scale matrix from the CORTO geometry and target mesh.
## Scale mat
!python /content/NeuS_thesis/custom_codes/preprocessing/scale_mat_builder.py \
  --geometry /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/geometry.json \
  --obj /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/jason2.obj  \
  --output \/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_spe3r/scale_mat.json \
  --margin 1.05

  #/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_spe3r/scale_mat.json

In [ ]:
# Convert the SPE3R-like dataset into the NeuS image/mask/camera layout.
## SPE3R to NeuS
!python /content/NeuS_thesis/custom_codes/preprocessing/spe3r_to_neus_data.py \
  --spe3r_dir /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_spe3r \
  --out_dir /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_neus \
  --scale_mat /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_spe3r/scale_mat.json

In [ ]:
# Run a consistency precheck on the generated NeuS dataset.
!python /content/NeuS_thesis/custom_codes/preprocessing/neus_precheck.py \
  --neus_dir /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_neus\
  --out /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_neus/output_precheck \
  --mesh /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/jason2.obj

### Realistic Light Filter

Filtering for the realistic-light sequence.


In [ ]:
# Select frames according to the phase-angle filter used for the realistic-light setup.
!python /content/NeuS_thesis/custom_codes/preprocessing/filter_by_phase_angle.py \
--geometry /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/geometry.json \
--output_dir /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src_filtered \
--threshold 120.0

In [ ]:
# Rename generated masks so their filenames match the format expected by later scripts.
# Rename to match previous scripts
import os

cartella = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/mask"

for i in range(100):
    vecchio_nome = os.path.join(cartella, f"{i:06d}.png")
    nuovo_nome = os.path.join(cartella, f"mask_{i:06d}_{i+1:04d}.png")

    if os.path.exists(vecchio_nome):
        os.rename(vecchio_nome, nuovo_nome)
        print(f"{vecchio_nome} -> {nuovo_nome}")
    else:
        print(f"File non trovato: {vecchio_nome}")

In [ ]:
# Build a filtered CORTO source folder using the accepted frame indices.
# Build CORTO_src
!python /content/NeuS_thesis/custom_codes/preprocessing/build_sun_filtered_dataset.py \
--accepted  /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src_filtered/accepted_frames.npy \
--images    /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/img \
--masks     /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/mask \
--geometry  /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/geometry.json \
--output    /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src_filtered

In [ ]:
# Check that each object in geometry.json has matching position and orientation lengths.
# Check geometry.json
import json

json_path = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src_filtered/geometry.json"

with open(json_path, "r") as f:
    data = json.load(f)

for obj_name, obj_data in data.items():
    pos_len = len(obj_data["position"])
    ori_len = len(obj_data["orientation"])

    print(f"{obj_name}:")
    print(f"  position    -> {pos_len}")
    print(f"  orientation -> {ori_len}")

    if pos_len == ori_len:
        print("  ✅ coerente\n")
    else:
        print("  ❌ mismatch\n")

# COLMAP

COLMAP reconstruction.


### Download

COLMAP installation commands.


In [ ]:
# Install COLMAP in the Colab runtime and verify that the executable is available.
# COLMAP
print("📦 Installazione COLMAP...")
!apt-get update -qq
!apt-get install -y colmap > /dev/null 2>&1

# Verifica installazione
!colmap -h | head -n 3
print("✅ COLMAP installato")

### Run

COLMAP execution command.


In [ ]:
# Run the custom COLMAP wrapper on the filtered realistic-light dataset.
DATASET="/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_spe3r_filtered"
OUT="/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output"
NEUS="/content/NeuS_thesis"
SAT=""

 #--match-type exhaustive_matcher --match-type sequential_matcher \

!xvfb-run -a python /content/NeuS_thesis/custom_codes/COLMAP_codes/colmap_runner.py \
    --spe3r-path "$DATASET" \
    --satellite "$SAT" \
    --output "$OUT" \
    --neus-path "$NEUS" \
    --match-type exhaustive_matcher

In [ ]:
# Inspect the reconstructed COLMAP sparse model with model_analyzer.
!colmap model_analyzer --path /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output/sparse/0

### Dataset

Conversion from COLMAP outputs to a NeuS dataset.


In [ ]:
# Convert COLMAP images.bin into sparse_txt/images.txt for registered-image ordering.
## image.txt builder
import struct
import numpy as np
import os

def read_next_bytes(fid, num_bytes, format_char_sequence, endian_character="<"):
    data = fid.read(num_bytes)
    return struct.unpack(endian_character + format_char_sequence, data)

def read_images_binary(path_to_model_file):
    images = {}
    with open(path_to_model_file, "rb") as fid:
        num_images = read_next_bytes(fid, 8, "Q")[0]

        for _ in range(num_images):
            binary_image_properties = read_next_bytes(fid, 64, "idddddddi")
            image_id = binary_image_properties[0]
            qvec = np.array(binary_image_properties[1:5])
            tvec = np.array(binary_image_properties[5:8])
            camera_id = binary_image_properties[8]

            # Nome immagine
            image_name = ""
            while True:
                char = fid.read(1).decode("utf-8")
                if char == "\x00":
                    break
                image_name += char

            num_points2D = read_next_bytes(fid, 8, "Q")[0]

            if num_points2D > 0:
                x_y_id_s = read_next_bytes(fid, num_points2D * 24, "ddq" * num_points2D)
                xys = np.array(x_y_id_s[0::3])
                yys = np.array(x_y_id_s[1::3])
                point3D_ids = np.array(x_y_id_s[2::3])
                xys = np.column_stack([xys, yys])
            else:
                xys = np.zeros((0, 2))
                point3D_ids = np.array([])

            images[image_id] = {
                "qvec": qvec,
                "tvec": tvec,
                "camera_id": camera_id,
                "name": image_name,
                "xys": xys,
                "point3D_ids": point3D_ids,
            }

    return images


def write_images_txt(images, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w") as f:
        f.write("# Image list with two lines of data per image:\n")
        f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, IMAGE_NAME\n")
        f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")

        for image_id in sorted(images.keys()):
            img = images[image_id]
            q = img["qvec"]
            t = img["tvec"]

            # Riga principale
            f.write(f"{image_id} {q[0]} {q[1]} {q[2]} {q[3]} "
                    f"{t[0]} {t[1]} {t[2]} {img['camera_id']} {img['name']}\n")

            # Riga punti 2D
            line = ""
            for (xy, pid) in zip(img["xys"], img["point3D_ids"]):
                line += f"{xy[0]} {xy[1]} {pid} "
            f.write(line.strip() + "\n")


# ==== USO ====

images_bin = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output/sparse/0/images.bin"
images_txt = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output/sparse_txt/images.txt"

images = read_images_binary(images_bin)
write_images_txt(images, images_txt)

print(f"✅ Convertito in: {images_txt}")
print(f"Numero immagini: {len(images)}")

In [ ]:
# Generate NeuS cameras and copy only images/masks registered by COLMAP.
# Camera builder
!python /content/NeuS_thesis/preprocess_custom_data/colmap_preprocess/gen_cameras_with_masks.py \
    /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit/colmap_output/images \
    /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit/colmap_output/masks \
    /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit/COLMAP_neus \
    /content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit/colmap_output


### Check

Dataset validation after COLMAP preprocessing.


In [ ]:
# Run a NeuS dataset precheck on the COLMAP-based dataset.
!python /content/NeuS_thesis/custom_codes/preprocessing/neus_precheck.py \
  --neus_dir "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Merged_Orbit_neus" \
  --out "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Merged_Orbit_neus/precheck_report" \
  --n_views 8 \
  --mesh "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_src/Osiris_rex.obj"




# Two orbits dataset union

Two-orbit dataset union workflow.


## Fusion and Plots

Alignment and diagnostic plotting for the two-orbit dataset.


In [ ]:
# Align the COLMAP reconstructions from the two orbits into the CORTO frame.
!python "/content/NeuS_thesis/custom_codes/COLMAP_codes/align_colmap_orbits_to_corto.py" \
  --orbit1-colmap "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/First_Orbit/colmap_output_pruned" \
  --orbit1-labels "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/First_Orbit/CORTO_spe3r_filtered/labels.json" \
  --orbit1-geometry "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/First_Orbit/CORTO_spe3r_filtered/geometry.json" \
  --orbit2-colmap "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output_pruned" \
  --orbit2-labels "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_spe3r_filtered/labels.json" \
  --orbit2-geometry "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_spe3r_filtered/geometry.json" \
  --run-ransac \
  --output "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Alignment_Pruning"

###Plot

Alignment diagnostic plots.


In [ ]:
# Generate diagnostic plots for the aligned COLMAP orbit reconstructions.
!python "/content/NeuS_thesis/custom_codes/COLMAP_codes/plot_alignment_diagnostics.py" \
  --alignment-root "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Alignment_Pruning" \
  --output "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Alignment_Pruning/plots" \
  --orbit1-full-geometry "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/First_Orbit/CORTO_src/geometry.json" \
  --orbit2-full-geometry "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/CORTO_src/geometry.json"

## Pruning

Pruning of the aligned COLMAP sparse model.


In [ ]:
# Prune the COLMAP sparse model to keep the registered/usable subset.
# Pruning
!python "/content/NeuS_thesis/custom_codes/COLMAP_codes/prune_colmap_model.py" \
  --input-root "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output" \
  --output-root "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output_pruned" \
  --exclude img000019 img000020 img000021 img000022 \
  --copy-assets \
  --overwrite

## Dataset

Construction of the combined NeuS dataset.


In [ ]:
# Build the combined two-orbit NeuS dataset from aligned poses, images, and masks.
!python "/content/NeuS_thesis/custom_codes/COLMAP_codes/build_combined_neus_dataset.py" \
  --orbit1-aligned-poses "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Alignment_Pruning/orbit1/aligned_poses_all_fit.json" \
  --orbit1-images "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/First_Orbit/colmap_output/images" \
  --orbit1-masks "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/First_Orbit/colmap_output/masks" \
  --orbit2-aligned-poses "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_80/Alignment_Pruning/orbit2/aligned_poses_all_fit.json" \
  --orbit2-images "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output_pruned/images" \
  --orbit2-masks "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Second_Orbit_80/colmap_output_pruned/masks" \
  --camera-json "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/First_Orbit/CORTO_spe3r_filtered/camera.json" \
  --scale-mat-json "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_spe3r/scale_mat.json" \
  --output "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_COLMAP/Merged_Orbit_neus" \
  --overwrite

# Training

NeuS training commands.


In [ ]:
# Install dependencies required by NeuS training and mesh extraction.
# Dependencies
!pip install opencv-python pyhocon icecream tqdm scipy PyMCubes trimesh

In [ ]:
# Update the NeuS config paths and training iteration count for a selected experiment.
# ==============================
# Colab cell: update .conf paths + end_iter
# ==============================

from pathlib import Path

def update_conf_params(conf_path,
                       base_exp_dir_value,
                       data_dir_value,
                       end_iter_value,
                       output_path=None):
    """
    Sostituisce nel file .conf le righe:
      base_exp_dir =
      data_dir =
      end_iter =
    con i valori passati in input.

    Se output_path è None, sovrascrive il file originale.
    """

    conf_path = Path(conf_path)
    if output_path is None:
        output_path = conf_path
    else:
        output_path = Path(output_path)

    if not conf_path.exists():
        raise FileNotFoundError(f"File .conf non trovato: {conf_path}")

    lines = conf_path.read_text(encoding="utf-8").splitlines()
    new_lines = []

    found_base = False
    found_data = False
    found_end_iter = False

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("base_exp_dir"):
            new_lines.append(f"base_exp_dir = {base_exp_dir_value}")
            found_base = True

        elif stripped.startswith("data_dir"):
            new_lines.append(f"data_dir = {data_dir_value}")
            found_data = True

        elif stripped.startswith("end_iter"):
            new_lines.append(f"end_iter = {end_iter_value}")
            found_end_iter = True

        else:
            new_lines.append(line)

    if not found_base:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'base_exp_dir'")
    if not found_data:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'data_dir'")
    if not found_end_iter:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'end_iter'")

    output_path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")
    print(f"[OK] File aggiornato salvato in: {output_path}")


# =========
# ESEMPIO
# =========

conf_path = "/content/NeuS_thesis/confs/long_test.conf"
base_exp_dir_value = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/04_CS/training"
data_dir_value = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_neus"
end_iter_value = 300000

update_conf_params(
    conf_path=conf_path,
    base_exp_dir_value=base_exp_dir_value,
    data_dir_value=data_dir_value,
    end_iter_value=end_iter_value
)

In [ ]:
# Run NeuS training with the configured experiment settings.
%cd /content/NeuS_thesis
!python exp_runner.py \
    --mode train \
    --conf ./confs/long_test.conf \
    --case jason2

### Loss Graph

TensorBoard loss plotting.


In [ ]:
# Load TensorBoard event files and extract scalar loss series from experiment logs.
from pathlib import Path
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


# ============================================================
# Root containing the 6 experiment folders
# ============================================================

EXPERIMENTS_ROOT = Path(
    "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments"
)

EVENT_FILE_PATTERN = "events.out.tfevents.*"


# ============================================================
# Find TensorBoard event files
# ============================================================

def find_event_files_by_exp(experiments_root):
    """
    Expected structure:

        Experiments/
            00_GT/
                training/logs/events.out.tfevents...
            01_something/
                training/logs/events.out.tfevents...
            ...

    Returns:
        dict: {experiment_name: [event_file_paths]}
    """
    experiments_root = Path(experiments_root)

    event_files_by_exp = {}

    for exp_dir in sorted([p for p in experiments_root.iterdir() if p.is_dir()]):
        logs_dir = exp_dir / "training" / "logs"

        if not logs_dir.exists():
            print(f"Skipping {exp_dir.name}: no training/logs folder")
            continue

        event_files = sorted(logs_dir.glob(EVENT_FILE_PATTERN))

        if not event_files:
            print(f"Skipping {exp_dir.name}: no TensorBoard event files")
            continue

        event_files_by_exp[exp_dir.name] = event_files

    return event_files_by_exp


# ============================================================
# Read available scalar tags
# ============================================================

def get_scalar_tags(event_files):
    tags = set()

    for file in event_files:
        ea = EventAccumulator(str(file), size_guidance={"scalars": 0})
        ea.Reload()
        tags.update(ea.Tags().get("scalars", []))

    return sorted(tags)


# ============================================================
# Read scalar values
# ============================================================

def read_scalar_from_event_file(event_file, tag):
    ea = EventAccumulator(str(event_file), size_guidance={"scalars": 0})
    ea.Reload()

    available_tags = ea.Tags().get("scalars", [])

    if tag not in available_tags:
        return pd.DataFrame(columns=["wall_time", "step", "value"])

    events = ea.Scalars(tag)

    return pd.DataFrame({
        "wall_time": [e.wall_time for e in events],
        "step": [e.step for e in events],
        "value": [e.value for e in events],
    })


def read_scalar_from_multiple_event_files(
    event_files,
    tag,
    deduplicate_by_step=True
):
    dfs = []

    for file in event_files:
        df = read_scalar_from_event_file(file, tag)

        if not df.empty:
            df["event_file"] = str(file)
            dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=["wall_time", "step", "value", "event_file"])

    df = pd.concat(dfs, ignore_index=True)

    # Important for resumed trainings / multiple event files
    df = df.sort_values(["step", "wall_time"])

    if deduplicate_by_step:
        df = df.drop_duplicates(subset="step", keep="last")

    df = df.sort_values("step").reset_index(drop=True)

    return df


# ============================================================
# Build dictionaries used by your plotting cell
# ============================================================

event_files_by_exp = find_event_files_by_exp(EXPERIMENTS_ROOT)

tags_by_exp = {
    name: get_scalar_tags(files)
    for name, files in event_files_by_exp.items()
}

print("Experiments found:")
for name, files in event_files_by_exp.items():
    print(f"- {name}: {len(files)} event file(s)")

print("\nAvailable tags:")
for name, tags in tags_by_exp.items():
    print(f"\n{name}")
    for tag in tags:
        print(f"  - {tag}")

In [ ]:
# Plot the selected TensorBoard scalar over training with smoothing and log scaling.
# ============================================================
# Plot selected TensorBoard tag: log scale + linear zoom
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

TAG_TO_PLOT = "Loss/loss"
SMOOTH_WINDOW = 75
DEDUPLICATE_BY_STEP = True

MIN_POSITIVE = 1e-8


def get_smoothed_series(df, smooth_window=25):
    y = df["value"].astype(float).copy()
    if smooth_window > 1:
        y = y.rolling(smooth_window, center=True, min_periods=1).mean()
    return y


# ------------------------------------------------------------
# 1) Full training, log scale
# ------------------------------------------------------------

plt.figure(figsize=(13, 5))

for name, files in event_files_by_exp.items():

    if TAG_TO_PLOT not in tags_by_exp[name]:
        print(f"Skipping {name}: tag '{TAG_TO_PLOT}' not found")
        continue

    df = read_scalar_from_multiple_event_files(
        files,
        TAG_TO_PLOT,
        deduplicate_by_step=DEDUPLICATE_BY_STEP
    )

    if df.empty:
        continue

    y = get_smoothed_series(df, SMOOTH_WINDOW)
    y = np.maximum(y, MIN_POSITIVE)

    plt.plot(df["step"], y, label=name)

plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel(TAG_TO_PLOT)
plt.title(f"{TAG_TO_PLOT} over training — log scale")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Validation

Validation and mesh extraction commands.


In [ ]:
# Update the NeuS config paths and iteration count for validation.
# ==============================
# Colab cell: update .conf paths + end_iter
# ==============================

from pathlib import Path

def update_conf_params(conf_path,
                       base_exp_dir_value,
                       data_dir_value,
                       end_iter_value,
                       output_path=None):
    """
    Sostituisce nel file .conf le righe:
      base_exp_dir =
      data_dir =
      end_iter =
    con i valori passati in input.

    Se output_path è None, sovrascrive il file originale.
    """

    conf_path = Path(conf_path)
    if output_path is None:
        output_path = conf_path
    else:
        output_path = Path(output_path)

    if not conf_path.exists():
        raise FileNotFoundError(f"File .conf non trovato: {conf_path}")

    lines = conf_path.read_text(encoding="utf-8").splitlines()
    new_lines = []

    found_base = False
    found_data = False
    found_end_iter = False

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("base_exp_dir"):
            new_lines.append(f"base_exp_dir = {base_exp_dir_value}")
            found_base = True

        elif stripped.startswith("data_dir"):
            new_lines.append(f"data_dir = {data_dir_value}")
            found_data = True

        elif stripped.startswith("end_iter"):
            new_lines.append(f"end_iter = {end_iter_value}")
            found_end_iter = True

        else:
            new_lines.append(line)

    if not found_base:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'base_exp_dir'")
    if not found_data:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'data_dir'")
    if not found_end_iter:
        raise ValueError("Nel file .conf non è stata trovata una riga che inizia con 'end_iter'")

    output_path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")
    print(f"[OK] File aggiornato salvato in: {output_path}")


# =========
# ESEMPIO
# =========

conf_path = "/content/NeuS_thesis/confs/long_test.conf"
base_exp_dir_value = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/04_CS/validation"
data_dir_value = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_neus"
end_iter_value = 300000

update_conf_params(
    conf_path=conf_path,
    base_exp_dir_value=base_exp_dir_value,
    data_dir_value=data_dir_value,
    end_iter_value=end_iter_value
)

In [ ]:
# Run NeuS mesh validation from a trained checkpoint.
# Mesh validation
%cd /content/NeuS_thesis
!python exp_runner.py \
    --mode validate_mesh \
    --conf ./confs/long_test.conf \
    --is_continue \
    --case Jason

In [ ]:
# Render validation images for selected camera indices.
import subprocess
from pathlib import Path

NEUS_DIR = Path("/content/NeuS_thesis")
CONF_PATH = NEUS_DIR / "confs/long_test.conf"
CASE_NAME = "Jason2"

IMG_INDICES = [0, 25, 50, 75]

for idx in IMG_INDICES:
    print(f"\n=== validate_image img_idx={idx} ===")

    cmd = [
        "python", "exp_runner.py",
        "--mode", "validate_image",
        "--conf", str(CONF_PATH),
        "--case", CASE_NAME,
        "--is_continue",
        "--img_idx", str(idx),
    ]

    subprocess.run(cmd, cwd=str(NEUS_DIR))

# Postprocessing

Postprocessing commands.


## Allineamento COLMAP -> GT

COLMAP-to-GT mesh alignment.


In [ ]:
# Apply a fixed COLMAP-to-GT transform to an exported validation mesh.
!python /content/NeuS_thesis/custom_codes/postprocessing/align_colmap_ply_to_gt.py \
  --input "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/05_RL_CP_SM/validation/meshes/00300000.ply" \
  --output "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/05_RL_CP_SM/results/GT_00300000.ply" \
  --matrix_values \
    -3.204025 2.331644 -4.228652 10.370105 \
    0.722389 5.249154 2.346992 -5.462369 \
    4.774538 0.770486 -3.192800 5.051340 \
    0.000000 0.000000 0.000000 1.000000


  # 03_CP
  #6.832738 0.175016 1.494359 -0.695939 \
  #-0.012087 6.955097 -0.759300 0.022482 \
  #-1.504524 0.738953 6.792674 -0.097791 \
  #0 0 0 1


## Single Stats

Single-checkpoint mesh statistics.


In [ ]:
# Generate colored point clouds showing pred-to-GT and GT-to-pred distances.
# Colormap
# "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_src/Osiris_rex.obj"
#"/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/jason2.obj"

!python /content/NeuS_thesis/custom_codes/postprocessing/colormap.py \
--gt_path "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_src/Osiris_rex.obj" \
--mesh_dir "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/05_RL_CP_SM/results"\
--checks GT_00300000.ply \
--n_points 50000 \
--seed 42 \
--out_dir "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/05_RL_CP_SM/results"

In [ ]:
# Compute mesh metrics for one predicted mesh against the GT mesh.
import os

# Mesh metrics
!python /content/NeuS_thesis/custom_codes/postprocessing/mesh_metrics.py \
--pred_mesh "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/05_RL_CP_SM/results/GT_00300000.ply" \
--gt_mesh "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_src/Osiris_rex.obj" \
--n 100000 \
--seed 42 \
--out_dir "/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/05_RL_CP_SM/results/GT_00300000"


#/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/jason2.obj
#/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_src/Osiris_rex.obj

#/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/03_CP/results/GT_00100000.ply

## Global Stats

Global metric aggregation over checkpoints.


In [ ]:
# Loop over checkpoints, validate meshes, compute metrics, and save metric arrays.
# ============================================================
# NeuS – Validation loop over all checkpoints
#
# Produces:
#   pred_to_gt_stats.npy       (N, 5)
#   gt_to_pred_stats.npy       (N, 5)
#   symchamfer.npy             (N, 1)
#   hausdorff_distance.npy     (N, 1)
#   hausdorff_p95_distance.npy (N, 1)
#   iterations.npy             (N,)
#
# IMPORTANT FIX:
#   Each iteration evaluates only the mesh generated for the
#   current checkpoint. Old meshes cannot be selected by mistake.
# ============================================================

import os
import json
import subprocess
import tempfile
from pathlib import Path

import numpy as np


# ─────────────────────────────────────────────────────────────
# USER SETTINGS
# ─────────────────────────────────────────────────────────────

NEUS_DIR = "/content/NeuS_thesis"
CONF_PATH = "./confs/long_test.conf"
CASE_NAME = "Jason2"

BASE_EXP_DIR = (
    "/content/drive/MyDrive/Tesi/neus/Ablation_test/"
    "Experiments/04_CS/validation"
)

DATA_DIR = (
    "/content/drive/MyDrive/Tesi/neus/Ablation_test/"
    "Datasets/Sat_complex/Light_simple/Mask_GT/"
    "Pose_GT/CORTO_neus"
)

GT_MESH = ( "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_complex/Light_simple/Mask_GT/Pose_GT/CORTO_src/jason2.obj"
)

OUT_DIR = (
    "/content/drive/MyDrive/Tesi/neus/Ablation_test/"
    "Experiments/04_CS/results"
)

MESH_METRICS_SCRIPT = (
    "/content/NeuS_thesis/custom_codes/postprocessing/"
    "mesh_metrics.py"
)

ALIGN_SCRIPT = (
    "/content/NeuS_thesis/custom_codes/postprocessing/"
    "align_colmap_ply_to_gt.py"
)

ITER_START = 10_000
ITER_END = 300_000
ITER_STEP = 10_000

APPLY_ALIGNMENT = False

ALIGN_MATRIX_FILE = None

ALIGN_MATRIX_VALUES = [
    -3.204025, 2.331644, -4.228652, 10.370105,
     0.722389, 5.249154,  2.346992, -5.462369,
     4.774538, 0.770486, -3.192800,  5.051340,
     0.000000, 0.000000,  0.000000,  1.000000,
]

# False: delete each generated validation mesh after computing metrics.
# True: retain the generated meshes in BASE_EXP_DIR / "meshes".
KEEP_VALIDATION_MESHES = False


# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

STAT_KEYS = ["mean", "std", "median", "p95", "max"]


def update_conf(
    conf_path: str,
    base_exp_dir: str,
    data_dir: str,
    end_iter: int,
):
    """
    Update the fields required by the validation run.
    """
    path = Path(conf_path)

    lines = path.read_text(encoding="utf-8").splitlines()
    new_lines = []

    found = {
        "base_exp_dir": False,
        "data_dir": False,
        "end_iter": False,
    }

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("base_exp_dir"):
            new_lines.append(f"base_exp_dir = {base_exp_dir}")
            found["base_exp_dir"] = True

        elif stripped.startswith("data_dir"):
            new_lines.append(f"data_dir = {data_dir}")
            found["data_dir"] = True

        elif stripped.startswith("end_iter"):
            new_lines.append(f"end_iter = {end_iter}")
            found["end_iter"] = True

        else:
            new_lines.append(line)

    missing = [key for key, value in found.items() if not value]

    if missing:
        raise KeyError(
            f"Fields not found in configuration file {path}: {missing}"
        )

    path.write_text(
        "\n".join(new_lines) + "\n",
        encoding="utf-8",
    )


def run(cmd: list[str], cwd: str | None = None):
    """
    Execute a command and stop immediately if it fails.
    """
    print("▶", " ".join(str(value) for value in cmd))

    return subprocess.run(
        cmd,
        cwd=cwd,
        check=True,
    )


def stats_from_json(data: dict, section: str) -> list[float]:
    """
    Extract the five ordered statistics expected by the output arrays.
    """
    values = data[section]

    return [
        float(values[key])
        for key in STAT_KEYS
    ]


def snapshot_meshes(meshes_dir: Path) -> dict[Path, int]:
    """
    Store the modification timestamp of each existing PLY mesh.
    """
    return {
        path.resolve(): path.stat().st_mtime_ns
        for path in meshes_dir.glob("*.ply")
    }


def find_current_mesh(
    meshes_dir: Path,
    iteration: int,
    meshes_before_validation: dict[Path, int],
) -> Path:
    """
    Select only the mesh generated or modified by the current validation.

    Standard NeuS naming is expected to be:
        00010000.ply
        00020000.ply
        ...
        00300000.ply

    If the standard path is missing, accept a fallback only when exactly
    one PLY file was newly created or modified during the current run.
    """
    expected_mesh = meshes_dir / f"{iteration:08d}.ply"

    if expected_mesh.exists():
        return expected_mesh

    fresh_candidates = []

    for path in meshes_dir.glob("*.ply"):
        resolved = path.resolve()
        previous_timestamp = meshes_before_validation.get(resolved)
        current_timestamp = path.stat().st_mtime_ns

        if (
            previous_timestamp is None
            or current_timestamp > previous_timestamp
        ):
            fresh_candidates.append(path)

    if len(fresh_candidates) == 1:
        print(
            "  [WARN] Expected mesh name not found. "
            f"Using the only new or modified mesh: "
            f"{fresh_candidates[0].name}"
        )

        return fresh_candidates[0]

    if not fresh_candidates:
        raise FileNotFoundError(
            f"No mesh was generated for checkpoint {iteration} "
            f"in {meshes_dir}."
        )

    raise RuntimeError(
        f"Ambiguous mesh selection for checkpoint {iteration}. "
        f"New or modified meshes: "
        f"{[path.name for path in fresh_candidates]}"
    )


def save_arrays(
    out_dir: Path,
    pred_to_gt_arr: np.ndarray,
    gt_to_pred_arr: np.ndarray,
    symchamfer_arr: np.ndarray,
    hausdorff_arr: np.ndarray,
    hausdorff_p95_arr: np.ndarray,
    iters_arr: np.ndarray,
):
    """
    Save the cumulative metric arrays after each completed checkpoint.
    """
    np.save(out_dir / "pred_to_gt_stats.npy", pred_to_gt_arr)
    np.save(out_dir / "gt_to_pred_stats.npy", gt_to_pred_arr)
    np.save(out_dir / "symchamfer.npy", symchamfer_arr)
    np.save(out_dir / "hausdorff_distance.npy", hausdorff_arr)
    np.save(
        out_dir / "hausdorff_p95_distance.npy",
        hausdorff_p95_arr,
    )
    np.save(out_dir / "iterations.npy", iters_arr)


# ─────────────────────────────────────────────────────────────
# Main loop
# ─────────────────────────────────────────────────────────────

iterations = list(
    range(
        ITER_START,
        ITER_END + 1,
        ITER_STEP,
    )
)

n_iters = len(iterations)

pred_to_gt_arr = np.full((n_iters, 5), np.nan)
gt_to_pred_arr = np.full((n_iters, 5), np.nan)
symchamfer_arr = np.full((n_iters, 1), np.nan)
hausdorff_arr = np.full((n_iters, 1), np.nan)
hausdorff_p95_arr = np.full((n_iters, 1), np.nan)

iters_arr = np.asarray(iterations, dtype=int)

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

meshes_dir = Path(BASE_EXP_DIR) / "meshes"
meshes_dir.mkdir(parents=True, exist_ok=True)

full_conf = str(Path(NEUS_DIR) / CONF_PATH)

for idx, iteration in enumerate(iterations):
    print("\n" + "=" * 70)
    print(
        f"  Checkpoint {iteration:>7d} / {ITER_END}   "
        f"({idx + 1}/{n_iters})"
    )
    print("=" * 70)

    update_conf(
        conf_path=full_conf,
        base_exp_dir=BASE_EXP_DIR,
        data_dir=DATA_DIR,
        end_iter=iteration,
    )

    # --------------------------------------------------------
    # Critical fix:
    # remove an old copy of the expected mesh before validation.
    # The current iteration must generate a fresh file.
    # --------------------------------------------------------

    expected_mesh = meshes_dir / f"{iteration:08d}.ply"

    if expected_mesh.exists():
        print(
            f"  Removing stale mesh before validation: "
            f"{expected_mesh}"
        )

        expected_mesh.unlink()

    meshes_before_validation = snapshot_meshes(meshes_dir)

    run(
        [
            "python",
            "exp_runner.py",
            "--mode",
            "validate_mesh",
            "--conf",
            CONF_PATH,
            "--case",
            CASE_NAME,
            "--is_continue",
        ],
        cwd=NEUS_DIR,
    )

    pred_mesh = find_current_mesh(
        meshes_dir=meshes_dir,
        iteration=iteration,
        meshes_before_validation=meshes_before_validation,
    )

    print(f"  Predicted mesh: {pred_mesh}")

    aligned_mesh = None

    try:
        # ----------------------------------------------------
        # Optional alignment
        # ----------------------------------------------------

        if APPLY_ALIGNMENT:
            with tempfile.NamedTemporaryFile(
                suffix=".ply",
                delete=False,
            ) as temp_file:
                aligned_mesh = Path(temp_file.name)

            align_cmd = [
                "python",
                ALIGN_SCRIPT,
                "--input",
                str(pred_mesh),
                "--output",
                str(aligned_mesh),
            ]

            if ALIGN_MATRIX_FILE is not None:
                align_cmd += [
                    "--matrix_file",
                    str(ALIGN_MATRIX_FILE),
                ]

            elif ALIGN_MATRIX_VALUES is not None:
                align_cmd += [
                    "--matrix_values",
                    *[str(value) for value in ALIGN_MATRIX_VALUES],
                ]

            else:
                raise ValueError(
                    "APPLY_ALIGNMENT=True, but no alignment matrix "
                    "was provided."
                )

            run(align_cmd)

            mesh_for_metrics = aligned_mesh

        else:
            mesh_for_metrics = pred_mesh

        # ----------------------------------------------------
        # Metric computation
        # ----------------------------------------------------

        with tempfile.TemporaryDirectory() as temp_metrics_dir:
            temp_metrics_dir = Path(temp_metrics_dir)

            run(
                [
                    "python",
                    MESH_METRICS_SCRIPT,
                    "--pred_mesh",
                    str(mesh_for_metrics),
                    "--gt_mesh",
                    GT_MESH,
                    "--out_dir",
                    str(temp_metrics_dir),
                ]
            )

            stats_path = temp_metrics_dir / "stats.json"

            if not stats_path.exists():
                raise FileNotFoundError(
                    f"stats.json not found in {temp_metrics_dir}."
                )

            data = json.loads(
                stats_path.read_text(encoding="utf-8")
            )

            pred_to_gt_arr[idx] = stats_from_json(
                data,
                "pred_to_gt_3d",
            )

            gt_to_pred_arr[idx] = stats_from_json(
                data,
                "gt_to_pred_3d",
            )

            symchamfer_arr[idx, 0] = float(
                data["symmetric_chamfer_3d"]
            )

            hausdorff_arr[idx, 0] = float(
                data["hausdorff_3d"]
            )

            hausdorff_p95_arr[idx, 0] = float(
                data["hausdorff_p95_3d"]
            )

            print(
                f"  symChamfer    = "
                f"{symchamfer_arr[idx, 0]:.6e}"
            )

            print(
                f"  Hausdorff     = "
                f"{hausdorff_arr[idx, 0]:.6e}"
            )

            print(
                f"  Hausdorff p95 = "
                f"{hausdorff_p95_arr[idx, 0]:.6e}"
            )

        # Save cumulative arrays after every successful iteration.
        save_arrays(
            out_dir=out_dir,
            pred_to_gt_arr=pred_to_gt_arr,
            gt_to_pred_arr=gt_to_pred_arr,
            symchamfer_arr=symchamfer_arr,
            hausdorff_arr=hausdorff_arr,
            hausdorff_p95_arr=hausdorff_p95_arr,
            iters_arr=iters_arr,
        )

    finally:
        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        if (
            not KEEP_VALIDATION_MESHES
            and pred_mesh.exists()
        ):
            pred_mesh.unlink()

        if (
            aligned_mesh is not None
            and aligned_mesh.exists()
        ):
            aligned_mesh.unlink()


# ─────────────────────────────────────────────────────────────
# Final report
# ─────────────────────────────────────────────────────────────

print("\n✅ All done!")

print(
    f"   pred_to_gt_stats  : "
    f"{pred_to_gt_arr.shape}  {STAT_KEYS}"
)

print(
    f"   gt_to_pred_stats  : "
    f"{gt_to_pred_arr.shape}  {STAT_KEYS}"
)

print(f"   symchamfer        : {symchamfer_arr.shape}")
print(f"   hausdorff         : {hausdorff_arr.shape}")
print(f"   hausdorff_p95     : {hausdorff_p95_arr.shape}")
print(f"   iterations        : {iters_arr.shape}")
print(f"   Saved to          : {out_dir}")

In [ ]:
# Plot aggregated mesh metrics and print the final comparison table.
# Grafici e Tabella
!python /content/NeuS_thesis/custom_codes/postprocessing/plot_mesh_metrics.py \
  --experiment "Exp 00 - SC=/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/00_GT/results" \
  --experiment "Exp 01 - RL=/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/01_RL/results" \
  --experiment "Exp 02 - SM=/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/02_SM/results" \
  --experiment "Exp 03 - CP=/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/03_CP/results" \
  --experiment "Exp 04 - CS=/content/drive/MyDrive/Tesi/neus/Ablation_test/Experiments/04_CS/results" \
  --out_dir "/content/drive/MyDrive/Tesi/neus/Ablation_test/Global_results/01_04"

## Debug

Optional debugging and mask-quality checks.


In [ ]:
# Compute IoU statistics between Blender masks and SAM masks to assess segmentation quality.
# ============================================================
# Mask IoU comparison between two folders
# ============================================================
# INPUT: modifica i due path qui sotto
FOLDER_BLENDER = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_simple/Mask_GT/Pose_GT/CORTO_neus/mask"   # maschere da Blender
FOLDER_SAM     = "/content/drive/MyDrive/Tesi/neus/Ablation_test/Datasets/Sat_simple/Light_realistic/Mask_Segmentation/Pose_GT/CORTO_src/mask"       # maschere da SAM
# ============================================================

import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from glob import glob

def load_mask(path):
    """Carica un'immagine come maschera binaria (True = foreground)."""
    img = np.array(Image.open(path).convert("L"))
    return img > 127

def compute_iou(mask_a, mask_b):
    """Calcola IoU tra due maschere binarie."""
    if mask_a.shape != mask_b.shape:
        # Ridimensiona b alla shape di a se necessario
        from PIL import Image as PILImage
        img_b = PILImage.fromarray(mask_b.astype(np.uint8) * 255)
        img_b = img_b.resize((mask_a.shape[1], mask_a.shape[0]), PILImage.NEAREST)
        mask_b = np.array(img_b) > 127

    intersection = np.logical_and(mask_a, mask_b).sum()
    union        = np.logical_or(mask_a, mask_b).sum()
    return intersection / union if union > 0 else 1.0

def get_sorted_files(folder):
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp")
    files = []
    for ext in exts:
        files.extend(glob(os.path.join(folder, ext)))
    return sorted(files)

# --- Carica i file ---
files_b = get_sorted_files(FOLDER_BLENDER)
files_s = get_sorted_files(FOLDER_SAM)

assert len(files_b) > 0, f"Nessun file trovato in {FOLDER_BLENDER}"
assert len(files_s) > 0, f"Nessun file trovato in {FOLDER_SAM}"
assert len(files_b) == len(files_s), (
    f"Numero di maschere diverso: Blender={len(files_b)}, SAM={len(files_s)}"
)

print(f"Trovate {len(files_b)} maschere per ciascun set.\n")

# --- Calcola IoU per ogni coppia ---
results = []
for fb, fs in zip(files_b, files_s):
    mb = load_mask(fb)
    ms = load_mask(fs)
    iou = compute_iou(mb, ms)
    # Pixel SAM in più / in meno rispetto a Blender
    sam_extra    = np.logical_and(ms, ~mb).sum()   # in SAM ma non in Blender
    blender_extra = np.logical_and(mb, ~ms).sum()  # in Blender ma non in SAM
    results.append({
        "frame":          os.path.basename(fb),
        "IoU":            iou,
        "SAM_extra_px":   int(sam_extra),
        "Blender_extra_px": int(blender_extra),
        "SAM_is_larger":  sam_extra > blender_extra,
    })

df = pd.DataFrame(results)

# --- Statistiche aggregate ---
print("=== Statistiche IoU ===")
print(df["IoU"].describe().round(4))
print(f"\nFrames con IoU < 0.95 : {(df['IoU'] < 0.95).sum()}")
print(f"Frames con IoU < 0.99 : {(df['IoU'] < 0.99).sum()}")
print(f"\nSAM più GRANDE di Blender (più pixel foreground): {df['SAM_is_larger'].sum()} / {len(df)} frames")
print(f"SAM più PICCOLA di Blender (più conservativa)   : {(~df['SAM_is_larger']).sum()} / {len(df)} frames")

# --- Plot distribuzione IoU ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df["IoU"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribuzione IoU")
axes[0].set_xlabel("IoU")
axes[0].set_ylabel("Conteggio frame")
axes[0].axvline(df["IoU"].mean(), color="red", linestyle="--", label=f"Media={df['IoU'].mean():.4f}")
axes[0].legend()

axes[1].plot(df["IoU"].values, color="steelblue", linewidth=0.8)
axes[1].set_title("IoU per frame")
axes[1].set_xlabel("Frame index")
axes[1].set_ylabel("IoU")
axes[1].axhline(df["IoU"].mean(), color="red", linestyle="--")

axes[2].plot(df["SAM_extra_px"].values,     label="Pixel solo in SAM",     color="green",  linewidth=0.8)
axes[2].plot(df["Blender_extra_px"].values, label="Pixel solo in Blender", color="orange", linewidth=0.8)
axes[2].set_title("Discrepanza ai bordi (pixel)")
axes[2].set_xlabel("Frame index")
axes[2].set_ylabel("Pixel")
axes[2].legend()

plt.tight_layout()
plt.savefig("mask_iou_comparison.png", dpi=150)
plt.show()

# --- Mostra i frame più problematici ---
print("\n=== Top 10 frame con IoU più bassa ===")
print(df.nsmallest(10, "IoU")[["frame", "IoU", "SAM_extra_px", "Blender_extra_px"]].to_string(index=False))